In [83]:
import os

os.environ["HSA_OVERRIDE_GFX_VERSION"] = "10.3.0"

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [84]:
print("PyTorch:", torch.__version__)
print("HIP:", torch.version.hip)
print("CUDA:", torch.version.cuda)
print("Доступна GPU:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Устройство:", torch.cuda.get_device_name(0))
    print("Архитектура:", torch.cuda.get_device_properties(0))

PyTorch: 2.14.1+rocm7.14
HIP: 7.14.60850
CUDA: None
Доступна GPU: True
Устройство: AMD Radeon RX 6750 XT
Архитектура: _CudaDeviceProperties(name='AMD Radeon RX 6750 XT', major=10, minor=3, gcnArchName='gfx1030', total_memory=12272MB, multi_processor_count=20, uuid=58580000-0000-0000-0000-000000000000, pci_bus_id=9, pci_device_id=0, pci_domain_id=0, L2_cache_size=3MB)


In [85]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

In [86]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

In [87]:
model = SimpleCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#device = "cpu"
model.to(device)

SimpleCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=6272, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)

In [88]:
num_epochs = 15

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Эпоха [{epoch+1}/{num_epochs}], Потери: {running_loss/len(train_loader):.4f}')

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    print(f'Точность на тестовом наборе: {accuracy:.2f}%')

Эпоха [1/15], Потери: 0.1058
Точность на тестовом наборе: 98.74%
Эпоха [2/15], Потери: 0.0406
Точность на тестовом наборе: 98.50%
Эпоха [3/15], Потери: 0.0288
Точность на тестовом наборе: 98.85%
Эпоха [4/15], Потери: 0.0213
Точность на тестовом наборе: 98.55%
Эпоха [5/15], Потери: 0.0181
Точность на тестовом наборе: 99.11%
Эпоха [6/15], Потери: 0.0129
Точность на тестовом наборе: 98.66%
Эпоха [7/15], Потери: 0.0143
Точность на тестовом наборе: 98.75%
Эпоха [8/15], Потери: 0.0119
Точность на тестовом наборе: 99.12%
Эпоха [9/15], Потери: 0.0110
Точность на тестовом наборе: 99.23%
Эпоха [10/15], Потери: 0.0070
Точность на тестовом наборе: 99.24%
Эпоха [11/15], Потери: 0.0082
Точность на тестовом наборе: 98.93%
Эпоха [12/15], Потери: 0.0085
Точность на тестовом наборе: 98.48%
Эпоха [13/15], Потери: 0.0079
Точность на тестовом наборе: 99.23%
Эпоха [14/15], Потери: 0.0056
Точность на тестовом наборе: 99.24%
Эпоха [15/15], Потери: 0.0055
Точность на тестовом наборе: 99.23%
